# TinyGPT v2 — Convert Mix A and Mix B to GGUF

This notebook converts the **two best SFT checkpoints** from the `stage4/sft` notebook into GGUF files for LM Studio. It does **not** train, fine-tune, quantize, or modify the source checkpoints.

## Required Kaggle input

Attach the saved output of your **new `stage4/sft` notebook**. It should contain:

- `stage4_sft_run_20261023/mixA/sft_mixA_best.pt`
- `stage4_sft_run_20261023/mixB/sft_mixB_best.pt`

The notebook searches `/kaggle/input` if the preferred paths differ, but stops if it cannot unambiguously identify the intended pair.

## Artifacts

The notebook writes only deliverables and reports into this notebook's own output directory:

`/kaggle/working/tinygpt_gguf_export_20261024/`

Expected deliverables:

- `TinyGPT-v2-Mix-A-F16.gguf`
- `TinyGPT-v2-Mix-B-F16.gguf`
- `conversion_manifest.json`
- `README_LM_STUDIO.txt`
- per-model conversion logs

Temporary Hugging Face model folders and the `llama.cpp` checkout are kept under `/kaggle/temp/`, not the saved output, to avoid bloating the Kaggle output with duplicate weights and source code.

## Important

- Enable **Internet** so the notebook can retrieve the GPT-2 tokenizer and `llama.cpp` converter.
- Use a fresh Kaggle session if dependency imports are already broken from a previous notebook.
- The conversion uses FP16 GGUF to keep conversion loss low. These models are small (~51M parameters), so FP16 files should be manageable without quantization.
- A successful conversion is not the same as proving that LM Studio loads it. This notebook checks GGUF signatures, file sizes, model architecture, and export round-tripping; loading/testing in LM Studio is the next step.

In [1]:
# CELL 1 — Install a compatible conversion environment and get llama.cpp.
# This intentionally does NOT install or replace PyTorch.
import os, sys, subprocess, shutil, json, time, hashlib, textwrap
from pathlib import Path

WORK_ROOT = Path('/kaggle/temp/tinygpt_gguf_work_20261024')
OUTPUT_DIR = Path('/kaggle/working/tinygpt_gguf_export_20261024')
HF_ROOT = WORK_ROOT / 'hf_models'
LLAMA_CPP_DIR = WORK_ROOT / 'llama.cpp'
LOG_DIR = OUTPUT_DIR / 'conversion_logs'
for p in (WORK_ROOT, HF_ROOT, OUTPUT_DIR, LOG_DIR):
    p.mkdir(parents=True, exist_ok=True)

# Pin Transformers/tokenizers to the known-compatible stack used for TinyGPT testing.
# Do not pin/install torch here: Kaggle's working PyTorch build must remain unchanged.
pip_packages = [
    'transformers==4.57.3',
    'tokenizers==0.22.1',
    'huggingface-hub==0.36.0',
    'safetensors',
    'sentencepiece',
    'protobuf',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pip_packages], check=True)

if not (LLAMA_CPP_DIR / 'convert_hf_to_gguf.py').is_file():
    if LLAMA_CPP_DIR.exists():
        shutil.rmtree(LLAMA_CPP_DIR)
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/ggml-org/llama.cpp.git', str(LLAMA_CPP_DIR)
    ], check=True)

if not (LLAMA_CPP_DIR / 'convert_hf_to_gguf.py').is_file():
    raise RuntimeError('llama.cpp checkout does not contain convert_hf_to_gguf.py')

print('Python:', sys.version.split()[0])
print('Temporary work:', WORK_ROOT)
print('Final output:', OUTPUT_DIR)
print('llama.cpp converter:', LLAMA_CPP_DIR / 'convert_hf_to_gguf.py')
print('Dependency setup completed. PyTorch was not explicitly reinstalled.')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.0 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.0 which is incompatible.
Cloning into '/kaggle/temp/tinygpt_gguf_work_20261024/llama.cpp'...


Python: 3.13.15
Temporary work: /kaggle/temp/tinygpt_gguf_work_20261024
Final output: /kaggle/working/tinygpt_gguf_export_20261024
llama.cpp converter: /kaggle/temp/tinygpt_gguf_work_20261024/llama.cpp/convert_hf_to_gguf.py
Dependency setup completed. PyTorch was not explicitly reinstalled.


In [2]:
# CELL 2 — Discover both intended SFT best checkpoints and define the exact GPT-2 architecture.
import os, re, json, torch
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
PREFERRED_RUN_FRAGMENT = 'stage4_sft_run_20261023'
PREFERRED_NOTEBOOK_FRAGMENT = 'stage4-sft'
PREFERRED = {
    'mixA': INPUT_ROOT / 'notebooks/fakesiddharth/stage4-sft/stage4_sft_run_20261023/mixA/sft_mixA_best.pt',
    'mixB': INPUT_ROOT / 'notebooks/fakesiddharth/stage4-sft/stage4_sft_run_20261023/mixB/sft_mixB_best.pt',
}

def discover_best_checkpoint(mix_name):
    filename = f'sft_{mix_name}_best.pt'
    preferred = PREFERRED[mix_name]
    if preferred.is_file():
        return preferred.resolve()

    hits = sorted(p.resolve() for p in INPUT_ROOT.rglob(filename) if p.is_file())
    # Prefer the clearly named output from the intended stage4/sft run.
    intended = [p for p in hits if PREFERRED_RUN_FRAGMENT in str(p) and PREFERRED_NOTEBOOK_FRAGMENT in str(p)]
    if len(intended) == 1:
        return intended[0]
    if len(intended) > 1:
        raise RuntimeError(f'Multiple intended {mix_name} checkpoints found:\n' + '\n'.join(map(str, intended)))
    notebook_hits = [p for p in hits if PREFERRED_NOTEBOOK_FRAGMENT in str(p)]
    if len(notebook_hits) == 1:
        return notebook_hits[0]
    if len(notebook_hits) > 1:
        raise RuntimeError(
            f'Multiple {mix_name} checkpoints exist under the stage4-sft input, and the preferred run path was absent.\n'
            + '\n'.join(map(str, notebook_hits))
            + '\nAttach only the intended stage4/sft output or update PREFERRED in Cell 2.'
        )
    if len(hits) == 1:
        return hits[0]
    if not hits:
        raise FileNotFoundError(
            f'Could not find {filename} under /kaggle/input. Attach the saved output of the new stage4/sft notebook.'
        )
    raise RuntimeError(
        f'Multiple {filename} files found and none identifies the intended run. Attach the correct stage4/sft output or update PREFERRED.\n'
        + '\n'.join(map(str, hits))
    )

CHECKPOINTS = {name: discover_best_checkpoint(name) for name in ('mixA', 'mixB')}
for name, path in CHECKPOINTS.items():
    print(f'{name}: {path} ({path.stat().st_size / (1024**2):.1f} MiB)')

# This matches the architecture in the completed Stage 4 SFT training notebook.
MODEL_CONFIG_DICT = {
    'vocab_size': 50257,
    'n_positions': 256,
    'n_ctx': 256,
    'n_embd': 512,
    'n_layer': 8,
    'n_head': 8,
    'resid_pdrop': 0.0,
    'embd_pdrop': 0.0,
    'attn_pdrop': 0.0,
    'bos_token_id': 50256,
    'eos_token_id': 50256,
    'pad_token_id': 50256,
    'use_cache': False,
}

print('PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print('CUDA GPUs:', torch.cuda.device_count())
else:
    print('CUDA unavailable; conversion can still run on CPU.')

mixA: /kaggle/input/notebooks/fakesiddharth/stage4-sft/stage4_sft_run_20261023/mixA/sft_mixA_best.pt (194.9 MiB)
mixB: /kaggle/input/notebooks/fakesiddharth/stage4-sft/stage4_sft_run_20261023/mixB/sft_mixB_best.pt (194.9 MiB)
PyTorch: 2.11.0+cu128
CUDA GPUs: 2


In [3]:
# CELL 3 — Rebuild each SFT model, verify strict weight compatibility, and export HF intermediates.
from transformers import GPT2Config, GPT2LMHeadModel, AutoTokenizer
import torch

# Exact training-time GPT-2 configuration. GPT-2 default settings (e.g. GELU variant,
# layer norm epsilon and tied token embeddings) are kept from the pinned Transformers implementation.
MODEL_CONFIG = GPT2Config(**MODEL_CONFIG_DICT)
PARAMETER_COUNT_EXPECTED = 51_082_752

TOKENIZER = AutoTokenizer.from_pretrained('gpt2', use_fast=True)
if len(TOKENIZER) != 50257 or TOKENIZER.eos_token_id != 50256:
    raise RuntimeError(
        f'Unexpected GPT-2 tokenizer: vocab={len(TOKENIZER)}, eos_token_id={TOKENIZER.eos_token_id}; '
        'expected vocab=50257 and EOS=50256.'
    )
TOKENIZER.pad_token = TOKENIZER.eos_token
TOKENIZER.bos_token = TOKENIZER.eos_token

# This template mirrors the SFT data prompt style: User: ...\nAssistant: ... <EOS>
# It improves the odds that LM Studio will apply the same role framing used during training.
TOKENIZER.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}{{ message['content'] }}\n"
    "{% elif message['role'] == 'user' %}User: {{ message['content'] }}\n"
    "{% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }}{{ eos_token }}\n"
    "{% endif %}{% endfor %}"
    "{% if add_generation_prompt %}Assistant:{% endif %}"
)


def unwrap_model_state(payload, label):
    if not isinstance(payload, dict):
        raise TypeError(f'{label}: checkpoint payload is {type(payload).__name__}, expected a state-dict mapping.')
    state = payload
    for key in ('model_state_dict', 'state_dict', 'model', 'model_weights', 'weights'):
        candidate = payload.get(key)
        if isinstance(candidate, dict) and candidate and all(torch.is_tensor(v) for v in candidate.values()):
            state = candidate
            break
    if not state or not all(torch.is_tensor(v) for v in state.values()):
        raise RuntimeError(f'{label}: could not identify a plain tensor state dict in the checkpoint.')
    if all(k.startswith('module.') for k in state):
        state = {k[len('module.'):]: v for k, v in state.items()}
    return state


def load_safe_checkpoint(path, label):
    # best.pt is a weights-only state dict. Prefer weights_only=True where supported.
    try:
        payload = torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        payload = torch.load(path, map_location='cpu')
    state = unwrap_model_state(payload, label)
    model = GPT2LMHeadModel(MODEL_CONFIG)
    model.load_state_dict(state, strict=True)
    model.eval()
    params = sum(p.numel() for p in model.parameters())
    if params != PARAMETER_COUNT_EXPECTED:
        raise RuntimeError(f'{label}: parameter count {params:,} does not match expected {PARAMETER_COUNT_EXPECTED:,}.')
    return model


def verify_hf_roundtrip(model, hf_dir, label):
    prompt = 'User: Hello.\nAssistant:'
    encoded = TOKENIZER(prompt, return_tensors='pt')
    with torch.inference_mode():
        reference = model(**encoded, use_cache=False).logits
    reloaded = GPT2LMHeadModel.from_pretrained(hf_dir, local_files_only=True).eval()
    with torch.inference_mode():
        candidate = reloaded(**encoded, use_cache=False).logits
    max_abs_diff = float((reference - candidate).abs().max().item())
    if max_abs_diff > 1e-5:
        raise RuntimeError(f'{label}: HF save/reload logits drifted (max abs diff {max_abs_diff:.3g}).')
    del reloaded, reference, candidate
    gc.collect()
    return max_abs_diff

import gc
HF_DIRS = {}
ROUNDTRIP_CHECKS = {}
for mix_name in ('mixA', 'mixB'):
    print(f'\n=== Exporting {mix_name} ===')
    hf_dir = HF_ROOT / ('TinyGPT-v2-Mix-A' if mix_name == 'mixA' else 'TinyGPT-v2-Mix-B')
    if hf_dir.exists():
        shutil.rmtree(hf_dir)
    hf_dir.mkdir(parents=True, exist_ok=True)
    model = load_safe_checkpoint(CHECKPOINTS[mix_name], mix_name)
    model.config.architectures = ['GPT2LMHeadModel']
    model.config.name_or_path = hf_dir.name
    model.save_pretrained(hf_dir, safe_serialization=True)
    TOKENIZER.save_pretrained(hf_dir)
    # Make conversion provenance explicit and include the expected prompt format.
    (hf_dir / 'TINYGPT_MODEL_INFO.txt').write_text(
        f'TinyGPT v2 {mix_name.upper()}\n'
        f'Source checkpoint: {CHECKPOINTS[mix_name]}\n'
        'Architecture: GPT-2 causal language model\nParameters: 51,082,752\n'
        'Context length: 256 tokens\n'
        'Tokenizer: GPT-2, vocab 50257, EOS id 50256\n'
        'Prompt style: User: <message>\\nAssistant: <response><EOS>\n',
        encoding='utf-8'
    )
    diff = verify_hf_roundtrip(model, hf_dir, mix_name)
    ROUNDTRIP_CHECKS[mix_name] = {'max_abs_logit_diff_after_hf_reload': diff, 'hf_dir': str(hf_dir)}
    HF_DIRS[mix_name] = hf_dir
    print(f'Checkpoint loaded strictly; params={sum(p.numel() for p in model.parameters()):,}; HF round-trip max logit diff={diff:.3g}')
    del model
    gc.collect()

print('\nBoth checkpoints have been exported to temporary Hugging Face model folders.')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


=== Exporting mixA ===
Checkpoint loaded strictly; params=51,082,752; HF round-trip max logit diff=0

=== Exporting mixB ===
Checkpoint loaded strictly; params=51,082,752; HF round-trip max logit diff=0

Both checkpoints have been exported to temporary Hugging Face model folders.


In [4]:
# CELL 4 — Convert both Hugging Face folders to FP16 GGUF using llama.cpp's GPT-2 converter.
import subprocess, sys, time

CONVERTER = LLAMA_CPP_DIR / 'convert_hf_to_gguf.py'
if not CONVERTER.is_file():
    raise FileNotFoundError(f'Expected converter not found: {CONVERTER}')

GGUF_PATHS = {
    'mixA': OUTPUT_DIR / 'TinyGPT-v2-Mix-A-F16.gguf',
    'mixB': OUTPUT_DIR / 'TinyGPT-v2-Mix-B-F16.gguf',
}
CONVERSION_LOGS = {}


def run_converter(mix_name, hf_dir, out_path):
    if out_path.exists():
        out_path.unlink()  # never mistake an old partial file for this run's output
    log_path = LOG_DIR / f'{mix_name}_conversion.log'
    cmd = [
        sys.executable, str(CONVERTER), str(hf_dir),
        '--outfile', str(out_path),
        '--outtype', 'f16',
        '--model-name', f'TinyGPT v2 {mix_name.upper()}',
    ]
    print(f'\n[{mix_name}] Running converter:')
    print(' '.join(cmd))
    start = time.time()
    with log_path.open('w', encoding='utf-8') as log:
        proc = subprocess.Popen(
            cmd, cwd=str(LLAMA_CPP_DIR), stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
        return_code = proc.wait()
    elapsed = time.time() - start
    CONVERSION_LOGS[mix_name] = str(log_path)
    if return_code != 0:
        tail = log_path.read_text(encoding='utf-8', errors='replace')[-6000:]
        raise RuntimeError(f'{mix_name} GGUF conversion failed (exit={return_code}). Last log lines:\n{tail}')
    if not out_path.is_file() or out_path.stat().st_size < 1_000_000:
        raise RuntimeError(f'{mix_name} conversion exited successfully but output is missing or implausibly small: {out_path}')
    with out_path.open('rb') as f:
        magic = f.read(4)
    if magic != b'GGUF':
        raise RuntimeError(f'{mix_name} output does not have the GGUF magic signature; found {magic!r}')
    print(f'[{mix_name}] GGUF created: {out_path} ({out_path.stat().st_size / (1024**2):.1f} MiB, {elapsed:.1f}s)')

for mix_name in ('mixA', 'mixB'):
    run_converter(mix_name, HF_DIRS[mix_name], GGUF_PATHS[mix_name])

print('\nBoth FP16 GGUF conversions completed and passed basic file-signature checks.')


[mixA] Running converter:
/usr/bin/python3 /kaggle/temp/tinygpt_gguf_work_20261024/llama.cpp/convert_hf_to_gguf.py /kaggle/temp/tinygpt_gguf_work_20261024/hf_models/TinyGPT-v2-Mix-A --outfile /kaggle/working/tinygpt_gguf_export_20261024/TinyGPT-v2-Mix-A-F16.gguf --outtype f16 --model-name TinyGPT v2 MIXA
INFO:hf-to-gguf:Loading model: TinyGPT-v2-Mix-A
INFO:hf-to-gguf:Model architecture: GPT2LMHeadModel
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:blk.0.attn_qkv.bias,      torch.float32 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_qkv.weight,    torch.float32 --> F16, shape = {512, 1536}
INFO:hf-to-gguf:blk.0.attn_output.bias,   torch.float32 --> F32, shape = {512}
INFO:hf-to-gguf:blk.0.attn_output.weight, torch.float32 --> F16, shape = {512, 512}
INFO:hf-to-gguf:blk.0.attn_norm.bias,     torch.float32 --> F32, shape = {512}
INFO:hf-to-gguf:blk.

In [5]:
# CELL 5 — Final manifest, checksums, handoff instructions, and deliverable audit.
from datetime import datetime, timezone
import hashlib, json


def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        while True:
            chunk = f.read(chunk_bytes)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    'project': 'TinyGPT v2',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'conversion': 'llama.cpp convert_hf_to_gguf.py',
    'output_type': 'F16 GGUF (not quantized)',
    'architecture': 'GPT-2 / GPT2LMHeadModel',
    'parameters': PARAMETER_COUNT_EXPECTED,
    'context_length': 256,
    'tokenizer': {'source': 'gpt2', 'vocab_size': 50257, 'eos_token_id': 50256},
    'prompt_format': 'User: <message>\\nAssistant: <response><EOS>',
    'input_checkpoints': {k: str(v) for k, v in CHECKPOINTS.items()},
    'hf_roundtrip_checks': ROUNDTRIP_CHECKS,
    'artifacts': {},
    'conversion_logs': CONVERSION_LOGS,
}

for mix_name, path in GGUF_PATHS.items():
    if not path.is_file() or path.stat().st_size < 1_000_000:
        raise RuntimeError(f'Missing or invalid expected deliverable: {path}')
    manifest['artifacts'][mix_name] = {
        'path': str(path),
        'filename': path.name,
        'bytes': path.stat().st_size,
        'mib': round(path.stat().st_size / (1024**2), 3),
        'sha256': sha256_file(path),
        'magic': 'GGUF',
    }

manifest_path = OUTPUT_DIR / 'conversion_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

readme = f'''TinyGPT v2 — GGUF conversion output
===================================

Files:
- {GGUF_PATHS['mixA'].name}  (Mix A, FP16 GGUF)
- {GGUF_PATHS['mixB'].name}  (Mix B, FP16 GGUF)
- conversion_manifest.json   (input paths, file sizes, SHA-256 hashes, verification)
- conversion_logs/           (llama.cpp conversion logs)

Architecture: GPT-2 causal LM, 51,082,752 parameters
Context length: 256 tokens
Tokenizer: GPT-2 vocabulary (50,257 tokens), EOS ID 50256

Prompt format used during SFT: User: <message>\\nAssistant: <response><EOS>

LM Studio handoff:
1. Download these .gguf files from this notebook's saved Kaggle output.
2. In LM Studio, import/open each local GGUF file.
3. Start with a context length of 256 (the model's trained context size).
4. If LM Studio does not automatically apply a compatible chat template, use a role template matching the prompt format above: user turns start with 'User: ', assistant turns with 'Assistant: ', and assistant turns end with the model EOS token.
5. Test the same short prompts on the original PyTorch model and GGUF model. This notebook does not claim behavioral equivalence until that comparison is run.

These are FP16 GGUFs to minimize conversion loss. No quantization or training was performed here.
'''
(OUTPUT_DIR / 'README_LM_STUDIO.txt').write_text(readme, encoding='utf-8')

checksums = ''.join(f"{manifest['artifacts'][k]['sha256']}  {manifest['artifacts'][k]['filename']}\n" for k in ('mixA', 'mixB'))
(OUTPUT_DIR / 'checksums.sha256').write_text(checksums, encoding='utf-8')

print('\nFINAL OUTPUT AUDIT')
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        print(f'{p.relative_to(OUTPUT_DIR)} | {p.stat().st_size / (1024**2):.2f} MiB')
print('\nManifest:', manifest_path)
print('Output directory:', OUTPUT_DIR)
print('Ready to save this Kaggle notebook version with outputs enabled.')


FINAL OUTPUT AUDIT
README_LM_STUDIO.txt | 0.00 MiB
TinyGPT-v2-Mix-A-F16.gguf | 99.48 MiB
TinyGPT-v2-Mix-B-F16.gguf | 99.48 MiB
checksums.sha256 | 0.00 MiB
conversion_logs/mixA_conversion.log | 0.01 MiB
conversion_logs/mixB_conversion.log | 0.01 MiB
conversion_manifest.json | 0.00 MiB

Manifest: /kaggle/working/tinygpt_gguf_export_20261024/conversion_manifest.json
Output directory: /kaggle/working/tinygpt_gguf_export_20261024
Ready to save this Kaggle notebook version with outputs enabled.


## After conversion

1. Run all cells from the beginning with the `stage4/sft` saved output attached.
2. Confirm that both `.gguf` files appear under this notebook's **Output** tab.
3. Save a Kaggle version with outputs enabled.
4. Download both `.gguf` files to your computer and import them into LM Studio.
5. Start with a 256-token context. Test several prompts against the PyTorch outputs before deciding whether the conversion behaves acceptably.

If the converter fails, inspect the relevant file under `conversion_logs/` before changing dependencies or retraining anything. The notebook intentionally keeps Mix A and Mix B separate and never modifies the source `.pt` checkpoints.